In [1]:
import tensorflow as tf 
from tensorflow.keras.models import load_model 
import pickle 
import pandas as pd 
import numpy as np 

In [2]:
#Load the trained model 
model = load_model('model.h5') 

#Load the encoder and scaler 
with open('label_encoder_gender.pkl' , 'rb') as file:
    label_encoder_gender = pickle.load(file) 
    
with open('onehot_encoder_geo.pkl' , 'rb') as file:
    onehot_encoder_geo = pickle.load(file) 
    
with open('scaler.pkl' , 'rb') as file:
    scaler = pickle.load(file)           

In [3]:
#Example input data 
input_data = {
    'CreditScore' : 600 , 
    'Geography' : 'France' , 
    'Gender' : 'Male' , 
    'Age' : 40 , 
    'Tenure' : 3 , 
    'Balance' : 60000 , 
    'NumOfProducts' : 2 , 
    'HasCrCard' : 1 , 
    'IsActiveMember' : 1 , 
    'EstimatedSalary' : 50000
}

In [4]:
#Convert input data to Dataframe 
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [5]:
#Encode Gender 
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [8]:
#One-hot encode 'Geography' 
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded , columns = onehot_encoder_geo.get_feature_names_out(['Geography']) , index = input_df.index)

In [9]:
#Drop original Geography column and concatenate encoded columns 
input_df = pd.concat([input_df.drop(columns = ['Geography']) , geo_encoded_df] , axis=1)

In [10]:
#Scale the features
input_scaled = scaler.transform(input_df)

In [11]:
#Make prediction 
prediction = model.predict(input_scaled) 

prediction_proba = prediction[0][0]
prediction_proba

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step


np.float32(0.031788733)

In [12]:
#Convert to binary prediction 
if prediction_proba > 0.5:
    print('The customer is likely to churn.') 

else:
    print('The customer is not likely to churn.')    

The customer is not likely to churn.
